# HAWQ-v2 Quantization Slider

Move the slider to pick an average bit-width. The notebook re-allocates
bits per layer using the HAWQ-v2 trace sensitivities, applies the bit-plan
to the QLinear blocks, and re-evaluates Top-1 on Tiny-ImageNet.

**You must run `experiment.ipynb` first** - this notebook loads
`checkpoint.pt` and `sensitivities.json` produced there.

In [ ]:
import sys
from pathlib import Path
import json

HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForImageClassification

import data, analyzer_v2, bit_allocator, quantizer, ptq

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f'Device: {device}')

## Restore model + sensitivities from experiment.ipynb

In [ ]:
ckpt = torch.load('checkpoint.pt', map_location=device, weights_only=False)
target_names = ckpt['target_names']
sensitivities = ckpt['sensitivities']
fp32_acc = ckpt['fp32_acc']
fp32_size_mb = ckpt['fp32_size_mb']

# Rebuild the model architecture and load fine-tuned weights.
model = AutoModelForImageClassification.from_pretrained(
    'facebook/deit-small-patch16-224',
    num_labels=200,
    ignore_mismatched_sizes=True,
)
quantizer.replace_linear_with_qlinear(model, target_names)
model.load_state_dict(ckpt['state_dict'])
model = model.to(device)

params_per_layer = ptq.collect_qlinear_param_counts(model)
print(f'Restored model with {len(params_per_layer)} QLinear blocks')
print(f'FP32 reference: {fp32_acc*100:.2f}%, {fp32_size_mb:.2f}MB')

## Validation loader (small subset for fast slider response)

In [ ]:
_, val_loader, _ = data.load_tiny_imagenet(
    batch_size=64,
    num_workers=2,
    image_size=224,
    val_subset=1000,        # smaller -> faster slider response
    augment_train=False,
)
print(f'Val batches: {len(val_loader)}')

## Pre-compute results across the whole slider range

Doing this once up front lets the slider react instantly. We compute
HAWQ-v2 *and* the matching uniform baseline for each integer bit-width.

In [ ]:
allocator = bit_allocator.HAWQv2BitAllocator(candidate_weight_bits=(8, 6, 4, 2))

BUDGETS = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]

hawq_cache = {}
for b in BUDGETS:
    bit_assign = allocator.allocate_by_budget(sensitivities, b)
    avg_b = ptq.average_bits(bit_assign, params_per_layer)
    ptq.apply_bit_assignment(model, bit_assign, quantize_activations=False)
    acc = ptq.evaluate(model, val_loader, device=device)
    size_mb = ptq.compute_effective_size(model, bit_assign)
    hawq_cache[b] = {
        'avg_bits': avg_b,
        'accuracy': acc,
        'size_mb': size_mb,
        'assignment': bit_assign,
    }
    print(f'HAWQ-v2 budget={b}: avg={avg_b:.2f} | acc={acc*100:.2f}% | size={size_mb:.2f}MB')

uniform_cache = {}
for bits in [2, 4, 6, 8]:
    bit_assign = allocator.allocate_uniform(sensitivities, bits)
    ptq.apply_bit_assignment(model, bit_assign, quantize_activations=False)
    acc = ptq.evaluate(model, val_loader, device=device)
    size_mb = ptq.compute_effective_size(model, bit_assign)
    uniform_cache[bits] = {
        'avg_bits': float(bits),
        'accuracy': acc,
        'size_mb': size_mb,
    }
    print(f'Uniform-{bits}: acc={acc*100:.2f}% | size={size_mb:.2f}MB')

## The slider

Moving the slider re-renders the bit-allocation bar chart and the metrics
panel. No re-evaluation happens here - everything was pre-computed above.

In [ ]:
slider = widgets.FloatSlider(
    value=6.0, min=min(BUDGETS), max=max(BUDGETS), step=0.5,
    description='Avg bits:', continuous_update=False,
    layout=widgets.Layout(width='600px'),
)
out = widgets.Output()

def render(change=None):
    b = slider.value
    info = hawq_cache.get(b)
    if info is None:
        # Snap to nearest precomputed budget.
        b = min(hawq_cache.keys(), key=lambda k: abs(k - b))
        info = hawq_cache[b]

    nearest_uni = min(uniform_cache.keys(), key=lambda k: abs(k - info['avg_bits']))
    uni = uniform_cache[nearest_uni]

    with out:
        clear_output(wait=True)
        print(f"=== HAWQ-v2 (budget={b}) ===")
        print(f'  avg bits  : {info["avg_bits"]:.2f}')
        print(f'  accuracy  : {info["accuracy"]*100:.2f}%   (FP32 = {fp32_acc*100:.2f}%)')
        print(f'  size      : {info["size_mb"]:.2f} MB     (FP32 = {fp32_size_mb:.2f} MB)')
        print(f'  compression : {fp32_size_mb / info["size_mb"]:.2f}x')
        print()
        print(f"=== Uniform-{nearest_uni} reference ===")
        print(f'  accuracy  : {uni["accuracy"]*100:.2f}%')
        print(f'  size      : {uni["size_mb"]:.2f} MB')
        print()
        gain = (info['accuracy'] - uni['accuracy']) * 100
        print(f'HAWQ-v2 gain over uniform-{nearest_uni}: {gain:+.2f} percentage points')

        assignment = info['assignment']
        names = list(assignment.keys())
        bits_list = [assignment[n] for n in names]

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        # Per-layer bit allocation
        colors = ['#d62728' if x <= 2 else '#ff7f0e' if x <= 4
                  else '#2ca02c' if x <= 6 else '#1f77b4' for x in bits_list]
        axes[0].bar(range(len(bits_list)), bits_list, color=colors)
        axes[0].set_xlabel('layer index (sensitivity-sorted)')
        axes[0].set_ylabel('bits')
        axes[0].set_title(f'Per-layer bit allocation (avg = {info["avg_bits"]:.2f})')
        axes[0].set_yticks([2, 4, 6, 8])
        axes[0].grid(True, axis='y', alpha=0.3)

        # Tradeoff curve with the current point highlighted
        budgets_sorted = sorted(hawq_cache.keys())
        accs = [hawq_cache[k]['accuracy'] * 100 for k in budgets_sorted]
        avgs = [hawq_cache[k]['avg_bits'] for k in budgets_sorted]
        u_avgs = [uniform_cache[k]['avg_bits'] for k in sorted(uniform_cache)]
        u_accs = [uniform_cache[k]['accuracy'] * 100 for k in sorted(uniform_cache)]

        axes[1].plot(u_avgs, u_accs, 'o-', label='Uniform PTQ', linewidth=2)
        axes[1].plot(avgs, accs, 's-', label='HAWQ-v2', linewidth=2)
        axes[1].axhline(fp32_acc * 100, color='gray', linestyle='--',
                        label=f'FP32 ({fp32_acc*100:.1f}%)')
        axes[1].scatter([info['avg_bits']], [info['accuracy'] * 100],
                         s=200, edgecolor='black', facecolor='gold', zorder=5)
        axes[1].set_xlabel('avg bit-width')
        axes[1].set_ylabel('Top-1 accuracy (%)')
        axes[1].set_title('Accuracy vs avg bit-width')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

slider.observe(render, names='value')
display(slider, out)
render()